# Notebook 1: Setup Environment - Spark + Nessie

**Objectif**: Initialiser l'environnement de travail et se connecter au catalogue de données Nessie.

## Concepts clés

- **Apache Spark**: Moteur de traitement distribué pour les données volumineuses
- **Project Nessie**: Catalogue de données avec contrôle de version (comme Git pour les données)
- **Apache Iceberg**: Format de table ACID avec time travel

## Architecture

```
Spark Application
        │
        ├── spark_catalog (catalogue par défaut)
        │
        └── nessie (notre catalogue versionné)
                │
                ├── bronze (données brutes)
                ├── silver (données nettoyées)
                └── gold (agrégats métier)
```

**Prérequis**: 
- Nessie démarré: `docker run -p 19120:19120 ghcr.io/projectnessie/nessie:latest`
- Données uploadées sur S3: `python scripts/upload_to_s3.py --all`

---
## INITIALISATION

In [1]:
# Configuration du chemin vers le projet
import sys
import os

project_root = os.path.abspath("..")
src_path = os.path.join(project_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

# Import des modules Spark et configuration
from lakehouse.spark_session import get_spark
from lakehouse.settings import AWS_BUCKET, NESSIE_URI

# Création de la session Spark
spark = get_spark("demo-setup")

print("=" * 60)
print("DÉMO LAKEHOUSE - INITIALISATION")
print("=" * 60)

DÉMO LAKEHOUSE - INITIALISATION


### Configuration de la branche Nessie main

In [2]:
# Configuration de la branche Nessie main
spark.conf.set("spark.sql.catalog.nessie.ref", "main")

# Création des namespaces (layers du medallion architecture)
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.bronze")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

print("Branche active: main")
print("Namespaces Nessie: bronze, silver, gold")
print(f"Bucket S3: {AWS_BUCKET}")

Branche active: main
Namespaces Nessie: bronze, silver, gold
Bucket S3: iceberg-nessie-lakehouse-demo


### Vérification des catalogues disponibles

In [3]:
print("=== Catalogues disponibles ===")
spark.sql("SHOW CATALOGS").show(truncate=False)

=== Catalogues disponibles ===
+-------------+
|catalog      |
+-------------+
|nessie       |
|spark_catalog|
+-------------+



### Vérification des namespaces Nessie

In [4]:
print("=== Namespaces Nessie ===")
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

=== Namespaces Nessie ===
+---------+
|namespace|
+---------+
|silver   |
|bronze   |
|gold     |
+---------+



### Vérification des tables existantes

In [6]:
print("=== Tables dans Bronze ===")
spark.sql("SHOW TABLES IN nessie.bronze").show(truncate=False)

print("=== Tables dans Silver ===")
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

print("=== Tables dans Gold ===")
spark.sql("SHOW TABLES IN nessie.gold").show(truncate=False)

=== Tables dans Bronze ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

=== Tables dans Silver ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+

=== Tables dans Gold ===
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



### Test de connexion S3 et vérification des fichiers batch

In [7]:
print(f"Bucket S3 configuré: {AWS_BUCKET}")
print("\n=== Fichiers batch disponibles sur S3 ===")

# Tester l'accès aux fichiers batch
batch_files = [
    "sales_batch_001.csv",
    "sales_batch_002.csv",
    "sales_batch_003.csv"
]

for batch_file in batch_files:
    batch_path = f"s3a://{AWS_BUCKET}/raw/batches/{batch_file}"
    try:
        test_df = spark.read.option("header", "true").csv(batch_path)
        count = test_df.count()
        print(f"  ✅ {batch_file}: {count:,} enregistrements")
    except Exception as e:
        print(f"  ❌ {batch_file}: Non accessible - {e}")

print("\nSi les fichiers ne sont pas accessibles, exécutez:")
print("  python scripts/upload_to_s3.py --all")

Bucket S3 configuré: iceberg-nessie-lakehouse-demo

=== Fichiers batch disponibles sur S3 ===
  ✅ sales_batch_001.csv: 3,364 enregistrements
  ✅ sales_batch_002.csv: 3,501 enregistrements
  ✅ sales_batch_003.csv: 3,500 enregistrements

Si les fichiers ne sont pas accessibles, exécutez:
  python scripts/upload_to_s3.py --all


### Affichage des branches Nessie

In [8]:
print("=== Branches Nessie ===")
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

=== Branches Nessie ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|8b73bcbb299d7689cf4632addf7dc07d4796e52af7c2c2c86ce0ede644fd7008|
+-------+----+----------------------------------------------------------------+



### Affichage de la branche active

In [15]:
print("=== Branche active ===")
spark.sql("SHOW REFERENCE IN nessie").show(truncate=False)

current_ref = spark.sql("SHOW REFERENCE IN nessie").first()[1]
print(f"Nous sommes sur la branche: {current_ref}")

=== Branche active ===
+-------+----+----------------------------------------------------------------+
|refType|name|hash                                                            |
+-------+----+----------------------------------------------------------------+
|Branch |main|8b73bcbb299d7689cf4632addf7dc07d4796e52af7c2c2c86ce0ede644fd7008|
+-------+----+----------------------------------------------------------------+

Nous sommes sur la branche: main


### Résumé de l'environnement

In [16]:
print("""
╔═════════════════════════════════════════════════════════╗
║         ENVIRONNEMENT LAKEHOUSE PRÊT                    ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Spark Session   : Configuré                            ║
║  Nessie Catalog  : Connecté                             ║
║  Iceberg Format  : Activé                               ║
║  S3 Storage      : Configuré                            ║
║                                                         ║
║  Namespaces:                                            ║
║    • nessie.bronze  (données brutes)                    ║
║    • nessie.silver  (données nettoyées)                 ║
║    • nessie.gold    (agrégats métier)                   ║
║                                                         ║
║  Branche active:                                        ║
║    • main (production)                                  ║
║                                                         ║
╚═════════════════════════════════════════════════════════╝
""")

print("→ Prochaine étape: Exécuter '02_ingestion_bronze.ipynb'")


╔═════════════════════════════════════════════════════════╗
║         ENVIRONNEMENT LAKEHOUSE PRÊT                    ║
╠═════════════════════════════════════════════════════════╣
║                                                         ║
║  Spark Session   : Configuré                            ║
║  Nessie Catalog  : Connecté                             ║
║  Iceberg Format  : Activé                               ║
║  S3 Storage      : Configuré                            ║
║                                                         ║
║  Namespaces:                                            ║
║    • nessie.bronze  (données brutes)                    ║
║    • nessie.silver  (données nettoyées)                 ║
║    • nessie.gold    (agrégats métier)                   ║
║                                                         ║
║  Branche active:                                        ║
║    • main (production)                                  ║
║                                      